# MentalDistress Phase 14: Final Locked-Test Evaluation

This is the one-time locked-test evaluation of the frozen primary systems. Merely
preparing this notebook does not authorize execution. Run it only after Phase 12
is complete, downloaded, and attached as a Kaggle input.

The notebook evaluates all ten seeds for A0 Flat, A1 Conditional CTPR, and A2
Always-route on original text and the two Phase 11 lexical stress tests. It never
trains, tunes, selects, drops a seed, changes a mask, or evaluates additional
controls. The test file is not discovered until every pre-opening gate passes.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import hashlib
import itertools
import json
import os
import platform
import random
import shutil
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from safetensors.torch import load_file
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before Phase 14"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / "phase14_final_locked_test_results"
OPENING_RECORD_PATH = OUTPUT_DIR / "phase14_test_opening_record.json"
if OPENING_RECORD_PATH.is_file():
    raise RuntimeError("Phase 14 test-opening record already exists in this session")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir()
CACHE_ROOT = WORKING_ROOT / "phase14_input_cache"
if CACHE_ROOT.exists():
    shutil.rmtree(CACHE_ROOT)
CACHE_ROOT.mkdir()

DEVICE = torch.device("cuda:0")
ALL_SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
SYSTEMS = ["A0_flat", "A1_conditional", "A2_always_route"]
VARIANTS = ["original", "direct_mask", "operational_mask"]
LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}
HARD_PAIRS = [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others",
]
K_FIXED = 3
CONDITIONAL_TAU = 0.10
ALWAYS_ROUTE_TAU = 1.00
TAU_GRID = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90, 1.00]
MAX_LENGTH = 512
BATCH_SIZE = 16
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 2029
ECE_BINS = 15
NLL_EPSILON = 1e-12
HIGH_CONFIDENCE_THRESHOLD = 0.90
EXPECTED_HOLM_FAMILY_SIZES = {
    "primary_original_a1": 10,
    "secondary_original_a2": 10,
    "robustness_masked": 40,
}
VALIDATION_FILENAME = "MentalDistress_validation_LOCKED.csv"
TEST_FILENAME = "MentalDistress_test_LOCKED.csv"
SUMMARY_FILENAME = "mentalroberta_full_10seed_summary.csv"
VALIDATION_DIGEST = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
PHASE8_DIGEST = "6d30115f9da6a1963b2fc74eb1b4c057ecb685ca7ae6dcf0c61d177bb3fa89e1"
PHASE9_DIGEST = "9db43d16d0acd7c0fb68129ee2c2e72b88ed3a4ff73c8df7a7ecfd80fcc68ba3"
PHASE10_DIGEST = "7d9e0065632e024d4f9aece380a72cefec2b2bc3e03e18bf51e076229a421ff3"
PHASE11_DIGEST = "f3a0ae1499d8cee01e370cadcf3b09ee3ea36bbd9f0eb3e0bd0428fe9abc1068"
PHASE12_DIGEST = "c0ed1d6af54ae0abf377c818fc44a66ff150eaf4e087dd0511ae527bca24adac"
PHASE12_MANIFEST_DIGEST = "afd3f2ee5bea407bcdbfeb604eeb0d0789ab1938fc433724a567617f617d1355"
ACCESSED_INPUTS = []

random.seed(20260823)
np.random.seed(20260823)
torch.manual_seed(20260823)
torch.cuda.manual_seed_all(20260823)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

assert len(ALL_SEEDS) == 10 and len(HARD_PAIRS) == K_FIXED
assert SYSTEMS == ["A0_flat", "A1_conditional", "A2_always_route"]
assert VARIANTS == ["original", "direct_mask", "operational_mask"]
print("CUDA:", torch.cuda.get_device_name(0))
print("Frozen systems:", SYSTEMS)
print("Frozen H/K/tau:", HARD_PAIRS, K_FIXED, CONDITIONAL_TAU)
print("Test data accessed: False")


CUDA: Tesla T4
Frozen systems: ['A0_flat', 'A1_conditional', 'A2_always_route']
Frozen H/K/tau: ['Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others'] 3 0.1
Test data accessed: False


## Required Kaggle inputs

Attach locked validation, ten model runs and their summary, Phase 8 through Phase
12 result packages, and the locked test dataset. Pre-opening discovery names only
non-test inputs. The exact test filename is resolved later inside the irreversible
opening cell.


In [2]:
def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )


def record_input(
    path, role, access_stage="preopening", known_sha256=None, **extra
):
    path = Path(path)
    row = {
        "access_order": len(ACCESSED_INPUTS) + 1,
        "access_stage": access_stage,
        "role": role,
        "path": str(path),
        "sha256": known_sha256 if known_sha256 is not None else sha256_file(path),
        "bytes": path.stat().st_size,
    }
    row.update(extra)
    ACCESSED_INPUTS.append(row)


def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError(f"Duplicate ZIP members: {archive_path}")
        for name in names:
            validate_zip_member(name)
            target = (destination / name).resolve()
            if not str(target).startswith(str(destination) + os.sep):
                raise ValueError(f"Unsafe ZIP member: {name}")
        archive.extractall(destination)


def verify_manifest(root):
    root = Path(root)
    manifest_path = root / "sha256_manifest.csv"
    frame = pd.read_csv(manifest_path)
    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if set(frame["path"]) != actual or len(frame) != len(actual):
        raise ValueError(f"Manifest membership mismatch: {root}")
    for row in frame.itertuples(index=False):
        path = root / row.path
        if path.stat().st_size != int(row.bytes):
            raise ValueError(f"Manifest byte mismatch: {row.path}")
        if sha256_file(path) != row.sha256:
            raise ValueError(f"Manifest hash mismatch: {row.path}")
    return frame


def choose_identical(paths, description):
    paths = sorted(set(map(Path, paths)), key=str)
    if not paths:
        raise FileNotFoundError(description)
    digests = {sha256_file(path) for path in paths}
    if len(digests) != 1:
        raise ValueError(f"Conflicting duplicate inputs for {description}")
    return paths[0]


def exact_non_test_file(filename, role, expected_digest=None):
    if filename == TEST_FILENAME:
        raise RuntimeError("Test discovery is forbidden before the opening event")
    candidates = sorted(INPUT_ROOT.rglob(filename), key=str)
    path = choose_identical(candidates, filename)
    if expected_digest is not None and sha256_file(path) != expected_digest:
        raise ValueError(f"SHA-256 mismatch: {filename}")
    record_input(path, role)
    return path


def materialize_result(dirname, archive_name, expected_digest=None):
    directories = sorted(
        [path for path in INPUT_ROOT.rglob(dirname) if path.is_dir()], key=str
    )
    archives = sorted(INPUT_ROOT.rglob(archive_name), key=str)
    archive_path = None
    if archives:
        archive_path = choose_identical(archives, archive_name)
        if expected_digest is not None and sha256_file(archive_path) != expected_digest:
            raise ValueError(f"Archive SHA-256 mismatch: {archive_name}")
        record_input(archive_path, dirname + "_archive")
    if directories:
        roots = sorted(directories, key=str)
        manifest_digests = {sha256_file(root / "sha256_manifest.csv") for root in roots}
        if len(manifest_digests) != 1:
            raise ValueError(f"Conflicting extracted roots: {dirname}")
        root = roots[0]
    else:
        if archive_path is None:
            raise FileNotFoundError(f"Attach {archive_name} or {dirname}")
        destination = CACHE_ROOT / dirname
        destination.mkdir()
        safe_extract(archive_path, destination)
        root = destination / dirname
    verify_manifest(root)
    record_input(root / "sha256_manifest.csv", dirname + "_manifest")
    return root


def pair_slug(pair_name):
    first, second = pair_name.split("|")
    return first.lower() + "__" + second.lower()


def discover_preopening_inputs():
    return {
        "validation": exact_non_test_file(
            VALIDATION_FILENAME, "locked_validation", VALIDATION_DIGEST
        ),
        "summary": exact_non_test_file(SUMMARY_FILENAME, "global_summary"),
        "phase8": materialize_result(
            "phase8_all_pair_specialists_results",
            "MentalDistress_Phase8_All_Pair_Specialists_Results.zip",
            PHASE8_DIGEST,
        ),
        "phase9": materialize_result(
            "phase9_routing_calibration_results",
            "MentalDistress_Phase9_Routing_Calibration_Results.zip",
            PHASE9_DIGEST,
        ),
        "phase10": materialize_result(
            "phase10_controls_allocation_results",
            "MentalDistress_Phase10_Controls_and_Allocation_Results.zip",
            PHASE10_DIGEST,
        ),
        "phase11": materialize_result(
            "phase11_mechanism_reliability_robustness_results",
            "MentalDistress_Phase11_Mechanism_Reliability_Robustness_Results.zip",
            PHASE11_DIGEST,
        ),
        "phase12": materialize_result(
            "phase12_efficiency_benchmarking_results",
            "MentalDistress_Phase12_Efficiency_Benchmarking_Results.zip",
            PHASE12_DIGEST,
        ),
    }


print("PASS: pre-opening discovery helpers initialized without test search")


PASS: pre-opening discovery helpers initialized without test search


## Embedded deterministic analysis core

The locally tested core supplies exact Phase 11-compatible masking, classification
metrics, stratified paired bootstrap, exact McNemar, Holm correction, archive
safety, and deterministic ZIP creation without local-module imports at Kaggle.


In [3]:
import hashlib
import math
import re
import zipfile
from pathlib import Path
from pathlib import PurePosixPath

import numpy as np


FIXED_ZIP_TIME = (2020, 1, 1, 0, 0, 0)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_pair(first, second, labels):
    labels = tuple(labels)
    if first == second or first not in labels or second not in labels:
        raise ValueError("Pair requires two distinct known labels")
    return "|".join(sorted((first, second), key=labels.index))


def validate_zip_member(member):
    path = PurePosixPath(str(member))
    if path.is_absolute() or not path.parts or ".." in path.parts:
        raise ValueError(f"Unsafe ZIP member: {member}")
    normalized = path.as_posix()
    if normalized in {"", "."}:
        raise ValueError(f"Unsafe ZIP member: {member}")
    return normalized


def confusion_matrix_array(y_true, y_pred, class_count=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if y_true.ndim != 1 or y_true.shape != y_pred.shape or len(y_true) == 0:
        raise ValueError("Metric vectors must be aligned and nonempty")
    if ((y_true < 0) | (y_true >= class_count)).any():
        raise ValueError("True label outside class range")
    if ((y_pred < 0) | (y_pred >= class_count)).any():
        raise ValueError("Predicted label outside class range")
    encoded = class_count * y_true + y_pred
    return np.bincount(encoded, minlength=class_count**2).reshape(
        class_count, class_count
    )


def classification_metrics(y_true, y_pred, class_count=5):
    confusion = confusion_matrix_array(y_true, y_pred, class_count)
    true_positive = np.diag(confusion).astype(float)
    support = confusion.sum(axis=1).astype(float)
    predicted = confusion.sum(axis=0).astype(float)
    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros(class_count, dtype=float),
        where=predicted != 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros(class_count, dtype=float),
        where=support != 0,
    )
    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros(class_count, dtype=float),
        where=(precision + recall) != 0,
    )
    represented = support > 0
    return {
        "accuracy": float(true_positive.sum() / confusion.sum()),
        "macro_f1": float(f1.mean()),
        "balanced_accuracy": float(recall[represented].mean()),
        "represented_class_count": int(represented.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
        "confusion": confusion,
    }


def correction_harm_counts(y_true, global_pred, final_pred, active):
    y_true = np.asarray(y_true, dtype=int)
    global_pred = np.asarray(global_pred, dtype=int)
    final_pred = np.asarray(final_pred, dtype=int)
    active = np.asarray(active, dtype=bool)
    if not (y_true.shape == global_pred.shape == final_pred.shape == active.shape):
        raise ValueError("Correction/harm vectors do not align")
    changed = active & (global_pred != final_pred)
    corrections = int((changed & (global_pred != y_true) & (final_pred == y_true)).sum())
    harms = int((changed & (global_pred == y_true) & (final_pred != y_true)).sum())
    return {
        "corrections": corrections,
        "harms": harms,
        "net_corrections": corrections - harms,
    }


def _word_pattern(word):
    return rf"(?<![A-Za-z]){re.escape(word)}(?![A-Za-z])"


def _wildcard_pattern(stem):
    return rf"(?<![A-Za-z]){re.escape(stem)}[A-Za-z]*(?![A-Za-z])"


def _phrase_pattern(phrase):
    joined = r"\s+".join(re.escape(word) for word in phrase.split())
    return rf"(?<![A-Za-z]){joined}(?![A-Za-z])"


def compile_lexical_patterns(mask_spec, mode):
    if mode not in {"direct", "operational"}:
        raise ValueError(f"Unknown lexical-mask mode: {mode}")
    flags = 0 if mask_spec.get("case_sensitive", False) else re.IGNORECASE
    entries = []
    if mode == "operational" and mask_spec.get("self_harm_pattern", False):
        entries.append(
            (
                "operational:self[- ]harm*",
                re.compile(
                    r"(?<![A-Za-z])self(?:-|\s+)harm[A-Za-z]*(?![A-Za-z])",
                    flags=flags,
                ),
            )
        )
    if mode == "operational":
        for phrase in sorted(
            mask_spec.get("operational_phrases", ()),
            key=lambda value: (-len(value), value),
        ):
            entries.append(
                (f"operational:{phrase}", re.compile(_phrase_pattern(phrase), flags=flags))
            )
    for stem in mask_spec.get("direct_wildcards", ()):
        entries.append((f"direct:{stem}*", re.compile(_wildcard_pattern(stem), flags=flags)))
    if mode == "operational":
        for stem in mask_spec.get("operational_wildcards", ()):
            entries.append(
                (f"operational:{stem}*", re.compile(_wildcard_pattern(stem), flags=flags))
            )
        for word in mask_spec.get("operational_exact_words", ()):
            entries.append((f"operational:{word}", re.compile(_word_pattern(word), flags=flags)))
    return tuple(entries)


def mask_text(text, mask_spec, mode):
    if not isinstance(text, str):
        raise TypeError("Lexical masking requires a string")
    mask_token = str(mask_spec.get("replacement_token", "<mask>"))
    candidates = []
    for priority, (cue_id, pattern) in enumerate(compile_lexical_patterns(mask_spec, mode)):
        for match in pattern.finditer(text):
            candidates.append(
                (match.start(), -(match.end() - match.start()), priority, match.end(), cue_id)
            )
    candidates.sort()
    selected = []
    current_end = -1
    for start, unused_length, unused_priority, end, cue_id in candidates:
        del unused_length, unused_priority
        if start < current_end:
            continue
        selected.append((start, end, cue_id))
        current_end = end
    if not selected:
        return {
            "text": text,
            "affected": False,
            "span_count": 0,
            "cue_ids": "",
            "original_characters": len(text),
            "masked_characters": len(text),
        }
    pieces = []
    cue_ids = []
    cursor = 0
    for start, end, cue_id in selected:
        pieces.extend((text[cursor:start], f" {mask_token} "))
        cursor = end
        if cue_id not in cue_ids:
            cue_ids.append(cue_id)
    pieces.append(text[cursor:])
    masked = re.sub(r"\s+", " ", "".join(pieces)).strip() or mask_token
    return {
        "text": masked,
        "affected": True,
        "span_count": len(selected),
        "cue_ids": ";".join(cue_ids),
        "original_characters": len(text),
        "masked_characters": len(masked),
    }


def class_stratified_bootstrap_indices(y_true, reps=5000, seed=2028):
    y_true = np.asarray(y_true, dtype=int)
    if y_true.ndim != 1 or len(y_true) == 0:
        raise ValueError("Bootstrap labels must be a nonempty vector")
    reps = int(reps)
    if reps <= 0:
        raise ValueError("Bootstrap reps must be positive")
    strata = [np.flatnonzero(y_true == label) for label in np.unique(y_true)]
    if any(len(stratum) == 0 for stratum in strata):
        raise ValueError("Bootstrap stratum is empty")
    rng = np.random.default_rng(int(seed))
    output = np.empty((reps, len(y_true)), dtype=int)
    for replicate in range(reps):
        sampled = [rng.choice(stratum, size=len(stratum), replace=True) for stratum in strata]
        output[replicate] = np.concatenate(sampled)
    return output


def paired_macro_f1_bootstrap(
    y_true,
    system_pred,
    flat_pred,
    class_count=5,
    reps=5000,
    seed=2028,
    alpha=0.05,
):
    y_true = np.asarray(y_true, dtype=int)
    system_pred = np.asarray(system_pred, dtype=int)
    flat_pred = np.asarray(flat_pred, dtype=int)
    if not (y_true.shape == system_pred.shape == flat_pred.shape):
        raise ValueError("Paired bootstrap vectors do not align")
    if not 0.0 < float(alpha) < 1.0:
        raise ValueError("alpha must lie between zero and one")
    observed = classification_metrics(y_true, system_pred, class_count)["macro_f1"]
    observed -= classification_metrics(y_true, flat_pred, class_count)["macro_f1"]
    indices = class_stratified_bootstrap_indices(y_true, reps=reps, seed=seed)
    deltas = np.empty(int(reps), dtype=float)
    for replicate, sampled in enumerate(indices):
        system_metric = classification_metrics(
            y_true[sampled], system_pred[sampled], class_count
        )["macro_f1"]
        flat_metric = classification_metrics(
            y_true[sampled], flat_pred[sampled], class_count
        )["macro_f1"]
        deltas[replicate] = system_metric - flat_metric
    lower, upper = np.quantile(deltas, [alpha / 2.0, 1.0 - alpha / 2.0])
    return {
        "observed_delta": float(observed),
        "ci_low": float(lower),
        "ci_high": float(upper),
        "bootstrap_mean_delta": float(deltas.mean()),
        "replicates": int(reps),
        "seed": int(seed),
    }


def exact_mcnemar(system_a_correct, system_b_correct):
    system_a_correct = np.asarray(system_a_correct, dtype=bool)
    system_b_correct = np.asarray(system_b_correct, dtype=bool)
    if system_a_correct.shape != system_b_correct.shape or system_a_correct.ndim != 1:
        raise ValueError("McNemar vectors do not align")
    a_correct_b_wrong = int((system_a_correct & ~system_b_correct).sum())
    a_wrong_b_correct = int((~system_a_correct & system_b_correct).sum())
    discordant = a_correct_b_wrong + a_wrong_b_correct
    if discordant == 0:
        p_value = 1.0
    else:
        smaller = min(a_correct_b_wrong, a_wrong_b_correct)
        lower_tail = sum(math.comb(discordant, value) for value in range(smaller + 1))
        p_value = min(1.0, 2.0 * lower_tail / (2**discordant))
    return {
        "a_correct_b_wrong": a_correct_b_wrong,
        "a_wrong_b_correct": a_wrong_b_correct,
        "discordant_total": discordant,
        "p_value": p_value,
    }


def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError("Holm p-values must be one finite vector")
    if ((values < 0.0) | (values > 1.0)).any():
        raise ValueError("Holm p-values must lie in [0, 1]")
    order = np.argsort(values, kind="stable")
    ranked = values[order]
    multipliers = np.arange(len(values), 0, -1, dtype=float)
    adjusted_ranked = np.maximum.accumulate(ranked * multipliers)
    adjusted_ranked = np.minimum(adjusted_ranked, 1.0)
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = adjusted_ranked
    return adjusted


def equal_seed_descriptives(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("Summary values must be one finite nonempty vector")
    first_quartile, third_quartile = np.quantile(values, [0.25, 0.75])
    return {
        "count": int(len(values)),
        "mean": float(values.mean()),
        "sd": float(values.std(ddof=1)) if len(values) > 1 else float("nan"),
        "median": float(np.median(values)),
        "q1": float(first_quartile),
        "q3": float(third_quartile),
        "iqr": float(third_quartile - first_quartile),
        "minimum": float(values.min()),
        "maximum": float(values.max()),
        "positive_fraction": float(np.mean(values > 0.0)),
    }


def deterministic_zip(source_dir, zip_path, archive_root):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    archive_root = validate_zip_member(archive_root)
    if not source_dir.is_dir():
        raise ValueError(f"Missing ZIP source directory: {source_dir}")
    with zipfile.ZipFile(
        zip_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for source_path in sorted(path for path in source_dir.rglob("*") if path.is_file()):
            relative = source_path.relative_to(source_dir).as_posix()
            archive_name = validate_zip_member(f"{archive_root}/{relative}")
            info = zipfile.ZipInfo(archive_name, date_time=FIXED_ZIP_TIME)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o600 << 16
            archive.writestr(info, source_path.read_bytes())
    return sha256_file(zip_path)


## Lightweight pre-input self-tests

These tests prove masking precedence, paired inference determinism, correction/harm
accounting, archive safety, and deterministic packaging before data are opened.


In [4]:
toy_spec = {
    "replacement_token": "<mask>",
    "case_sensitive": False,
    "direct_wildcards": ["depress", "anxi"],
    "operational_wildcards": ["sad"],
    "operational_exact_words": ["dead"],
    "operational_phrases": ["kill myself"],
    "self_harm_pattern": True,
    "operational_is_cumulative": True,
}
assert mask_text("DEPRESSED anxiety", toy_spec, "direct")["text"] == "<mask> <mask>"
assert mask_text("self-harming", toy_spec, "operational")["text"] == "<mask>"
toy_y = np.repeat(np.arange(5), 2)
toy_flat = np.array([0, 1, 1, 1, 2, 0, 3, 4, 4, 4])
toy_system = np.array([0, 0, 1, 1, 2, 2, 3, 4, 4, 4])
toy_boot = paired_macro_f1_bootstrap(
    toy_y, toy_system, toy_flat, reps=100, seed=9
)
assert toy_boot["observed_delta"] > 0 and toy_boot["replicates"] == 100
assert np.allclose(holm_adjust([0.01, 0.04, 0.03, 0.20]), [0.04, 0.09, 0.09, 0.20])
assert exact_mcnemar(toy_y == toy_system, toy_y == toy_system)["p_value"] == 1.0
helper_dir = WORKING_ROOT / "phase14_helper_test"
helper_dir.mkdir()
(helper_dir / "payload.txt").write_text("phase14\n", encoding="utf-8")
first_zip = WORKING_ROOT / "phase14_helper_one.zip"
second_zip = WORKING_ROOT / "phase14_helper_two.zip"
assert deterministic_zip(helper_dir, first_zip, "results") == deterministic_zip(
    helper_dir, second_zip, "results"
)
shutil.rmtree(helper_dir)
first_zip.unlink()
second_zip.unlink()
print("PASS: lightweight Phase 14 core tests")


PASS: lightweight Phase 14 core tests


## Pre-opening integrity gate

The notebook verifies Phase 8–12 artifacts, the supplied provenance limitation,
all checkpoints, all 100 heads, and derives the exact 30 selected heads from the
frozen allocation. Any disagreement stops while test access remains false.


In [5]:
discovered = discover_preopening_inputs()
VALIDATION_PATH = discovered["validation"]
SUMMARY_PATH = discovered["summary"]
PHASE8_ROOT = discovered["phase8"]
PHASE9_ROOT = discovered["phase9"]
PHASE10_ROOT = discovered["phase10"]
PHASE11_ROOT = discovered["phase11"]
PHASE12_ROOT = discovered["phase12"]
UPSTREAM_ARTIFACTS = {
    "phase8_archive_sha256": PHASE8_DIGEST,
    "phase8_manifest_sha256": sha256_file(PHASE8_ROOT / "sha256_manifest.csv"),
    "phase9_archive_sha256": PHASE9_DIGEST,
    "phase9_manifest_sha256": sha256_file(PHASE9_ROOT / "sha256_manifest.csv"),
    "phase10_archive_sha256": PHASE10_DIGEST,
    "phase10_manifest_sha256": sha256_file(PHASE10_ROOT / "sha256_manifest.csv"),
    "phase11_archive_sha256": PHASE11_DIGEST,
    "phase11_manifest_sha256": sha256_file(PHASE11_ROOT / "sha256_manifest.csv"),
    "phase12_archive_sha256": PHASE12_DIGEST,
    "phase12_manifest_sha256": sha256_file(PHASE12_ROOT / "sha256_manifest.csv"),
}
assert UPSTREAM_ARTIFACTS["phase12_manifest_sha256"] == PHASE12_MANIFEST_DIGEST
validation = pd.read_csv(VALIDATION_PATH)
assert len(validation) == 1490 and validation["original_row"].is_unique
assert {"original_row", "text", "label"}.issubset(validation.columns)
assert set(validation["label"]) == set(LABELS)
y_validation = validation["label"].map(LABEL_TO_ID).to_numpy(dtype=int)

phase8_protocol = json.loads((PHASE8_ROOT / "phase8_protocol.json").read_text())
assert phase8_protocol["head_count"] == 100
assert phase8_protocol["model_seeds"] == ALL_SEEDS
assert phase8_protocol["phase5_frozen_H"] == HARD_PAIRS
assert phase8_protocol["phase5_frozen_K"] == K_FIXED
assert phase8_protocol["representation"] == "final_hidden_state_token_0"
assert phase8_protocol["encoder_frozen"] is True
assert phase8_protocol["test_accessed"] is False
verified_heads = 0
verified_selected_heads = 0
FROZEN_SELECTED_HEADS = []
for model_seed in ALL_SEEDS:
    for pair_name in phase8_protocol["pairs"]:
        head_dir = PHASE8_ROOT / "heads" / f"seed_{model_seed}" / pair_slug(pair_name)
        metadata = json.loads((head_dir / "metadata.json").read_text())
        head_path = head_dir / "head.safetensors"
        assert metadata["model_seed"] == model_seed and metadata["pair"] == pair_name
        assert metadata["hidden_size"] == 768 and metadata["encoder_frozen"] is True
        assert metadata["test_accessed"] is False
        assert sha256_file(head_path) == metadata["head_sha256"]
        verified_heads += 1
        if pair_name in HARD_PAIRS:
            FROZEN_SELECTED_HEADS.append({
                "model_seed": model_seed,
                "pair": pair_name,
                "relative_path": head_path.relative_to(PHASE8_ROOT).as_posix(),
                "sha256": metadata["head_sha256"],
            })
            verified_selected_heads += 1
assert verified_heads == 100 and verified_selected_heads == 30
assert len(FROZEN_SELECTED_HEADS) == len(ALL_SEEDS) * K_FIXED

phase9 = json.loads((PHASE9_ROOT / "phase9_selected_config.json").read_text())
assert phase9["H"] == HARD_PAIRS and phase9["K_fixed"] == K_FIXED
assert phase9["tau_grid"] == TAU_GRID and phase9["selected_tau"] == CONDITIONAL_TAU
assert phase9["selected_mode"] == "specialist_routing" and phase9["test_accessed"] is False
phase10_protocol = json.loads((PHASE10_ROOT / "phase10_protocol.json").read_text())
phase10_execution = json.loads((PHASE10_ROOT / "phase10_execution_summary.json").read_text())
assert phase10_protocol["model_seeds"] == ALL_SEEDS and phase10_protocol["labels"] == LABELS
assert phase10_protocol["H"] == HARD_PAIRS and phase10_protocol["K"] == K_FIXED
assert phase10_protocol["frozen_tau"] == CONDITIONAL_TAU
assert phase10_protocol["test_accessed"] is False
assert phase10_execution["status"] == "PASS"
assert phase10_execution["phase9_reproduction_pass_count"] == 10
assert phase10_execution["test_accessed"] is False
MASK_SPEC_PATH = PHASE11_ROOT / "phase11_lexical_mask_spec.json"
MASK_SPEC = json.loads(MASK_SPEC_PATH.read_text(encoding="utf-8"))
phase11_protocol = json.loads((PHASE11_ROOT / "phase11_protocol.json").read_text())
phase11_execution = json.loads((PHASE11_ROOT / "phase11_execution_summary.json").read_text())
assert MASK_SPEC["version"] == "phase11-frozen-v1" and MASK_SPEC["test_accessed"] is False
assert phase11_protocol["model_seeds"] == ALL_SEEDS
assert phase11_execution["status"] == "PASS" and phase11_execution["test_accessed"] is False
LEXICAL_MASK_SPEC_DIGEST = sha256_file(MASK_SPEC_PATH)
phase12_execution = json.loads((PHASE12_ROOT / "phase12_execution_summary.json").read_text())
assert phase12_execution["status"] == "PASS"
assert phase12_execution["phase10_reproduction_pass_count"] == 10
assert phase12_execution["batch1_raw_rows"] == 230400
assert phase12_execution["throughput_raw_rows"] == 900
assert phase12_execution["vram_worker_pass_count"] == 30
assert phase12_execution["test_accessed"] is False

global_summary = pd.read_csv(SUMMARY_PATH)
assert sorted(global_summary["seed"].astype(int).tolist()) == ALL_SEEDS
assert global_summary["status"].astype(str).eq("PASS").all()
assert global_summary["test_accessed"].astype(str).str.lower().eq("false").all()
MODEL_DIRS = {}
GLOBAL_RUNS = {}
FROZEN_MODEL_RUNS = []
for row in global_summary.itertuples(index=False):
    model_seed = int(row.seed)
    candidates = [
        path for path in INPUT_ROOT.rglob(str(row.run_id))
        if path.is_dir()
        and (path / "models/best_model/model.safetensors").is_file()
        and (path / "artifacts/validation_predictions.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(str(row.run_id))
    manifest_hashes = {sha256_file(path / "models/sha256_manifest.csv") for path in candidates}
    if len(manifest_hashes) != 1:
        raise ValueError(f"Conflicting model run: {row.run_id}")
    run_dir = sorted(candidates, key=str)[0]
    verify_manifest(run_dir / "models")
    verify_manifest(run_dir / "artifacts")
    model_manifest_digest = sha256_file(run_dir / "models/sha256_manifest.csv")
    artifact_manifest_digest = sha256_file(run_dir / "artifacts/sha256_manifest.csv")
    FROZEN_MODEL_RUNS.append({
        "model_seed": model_seed,
        "run_id": str(row.run_id),
        "model_manifest_sha256": model_manifest_digest,
        "artifact_manifest_sha256": artifact_manifest_digest,
    })
    MODEL_DIRS[model_seed] = run_dir / "models/best_model"
    GLOBAL_RUNS[model_seed] = run_dir
assert sorted(MODEL_DIRS) == ALL_SEEDS
assert len(FROZEN_MODEL_RUNS) == len(ALL_SEEDS)
print("PASS: Phase 8-12 pre-opening integrity verified")
print("Test data accessed: False")


PASS: Phase 8-12 pre-opening integrity verified
Test data accessed: False


## Frozen model and routing implementation

Metadata controls specialist label mapping. A0 returns before routing preparation.
A1 and A2 share the frozen global representation and execute at most one matched
specialist per routed row.


In [6]:
def load_selected_heads(model_seed, device=DEVICE):
    weights, biases, class_maps = [], [], []
    for pair_name in HARD_PAIRS:
        head_dir = PHASE8_ROOT / "heads" / f"seed_{model_seed}" / pair_slug(pair_name)
        metadata = json.loads((head_dir / "metadata.json").read_text())
        head_path = head_dir / "head.safetensors"
        assert sha256_file(head_path) == metadata["head_sha256"]
        state = load_file(str(head_path), device="cpu")
        assert tuple(state["classifier.weight"].shape) == (2, 768)
        assert tuple(state["classifier.bias"].shape) == (2,)
        weights.append(state["classifier.weight"].float())
        biases.append(state["classifier.bias"].float())
        class_maps.append([
            LABEL_TO_ID[metadata["class_a"]], LABEL_TO_ID[metadata["class_b"]]
        ])
    lookup = torch.full((len(LABELS), len(LABELS)), -1, dtype=torch.long)
    for index, pair_name in enumerate(HARD_PAIRS):
        first, second = pair_name.split("|")
        a, b = LABEL_TO_ID[first], LABEL_TO_ID[second]
        lookup[a, b] = index
        lookup[b, a] = index
    return {
        "weights": torch.stack(weights).to(device),
        "biases": torch.stack(biases).to(device),
        "class_maps": torch.tensor(class_maps, dtype=torch.long, device=device),
        "pair_lookup": lookup.to(device),
    }


def global_forward(model, batch):
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        encoded = model.roberta(**batch, return_dict=True)
        sequence = encoded.last_hidden_state
        logits = model.classifier(sequence)
        return sequence, logits, torch.argmax(logits, dim=1)


def route_cached(sequence, logits, global_prediction, heads, system_id):
    if system_id == "A0_flat":
        return {
            "prediction": global_prediction,
            "global_prediction": global_prediction,
        }
    if system_id == "A1_conditional":
        tau = CONDITIONAL_TAU
    elif system_id == "A2_always_route":
        tau = ALWAYS_ROUTE_TAU
    else:
        raise KeyError(f"Unknown frozen system_id: {system_id}")
    probabilities = torch.softmax(logits, dim=1)
    top_values, top_ids = torch.topk(probabilities, k=2, dim=1)
    margin = top_values[:, 0] - top_values[:, 1]
    pair_index = heads["pair_lookup"][top_ids[:, 0], top_ids[:, 1]]
    active = (pair_index >= 0) & (margin <= tau)
    prediction = global_prediction.clone()
    if int(active.sum().item()):
        selected = pair_index[active]
        features = sequence[active, 0, :].unsqueeze(2)
        pair_logits = torch.bmm(heads["weights"][selected], features).squeeze(2)
        pair_logits = pair_logits + heads["biases"][selected]
        local = torch.argmax(pair_logits, dim=1)
        prediction[active] = heads["class_maps"][selected, local]
    return {
        "prediction": prediction,
        "global_prediction": global_prediction,
        "probabilities": probabilities,
        "top2": top_ids,
        "margin": margin,
        "active": active,
        "pair_index": pair_index,
    }


def forward_system(model, heads, batch, system_id):
    sequence, logits, global_prediction = global_forward(model, batch)
    return route_cached(sequence, logits, global_prediction, heads, system_id)


def forward_all_systems(model, heads, batch):
    sequence, logits, global_prediction = global_forward(model, batch)
    outputs = {
        system_id: route_cached(
            sequence, logits, global_prediction, heads, system_id
        ) for system_id in SYSTEMS
    }
    outputs["global_probabilities"] = torch.softmax(logits, dim=1)
    return outputs


def gpu_batch(tokenizer, texts):
    encoded = tokenizer(
        list(texts), padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt"
    )
    return {name: tensor.to(DEVICE) for name, tensor in encoded.items()}


# Lightweight GPU routing-contract tests. These do not use development or test data.
class _ToyEncoder:
    def __init__(self, sequence):
        self.sequence = sequence

    def __call__(self, **unused):
        return type("ToyEncoderOutput", (), {
            "last_hidden_state": self.sequence
        })()


class _ToyClassifier:
    def __init__(self, logits):
        self.logits = logits

    def __call__(self, unused_sequence):
        return self.logits


class _ToyModel:
    def __init__(self, sequence, logits):
        self.roberta = _ToyEncoder(sequence)
        self.classifier = _ToyClassifier(logits)


toy_sequence = torch.zeros((1, 2, 768), device=DEVICE)
toy_weights = torch.zeros((K_FIXED, 2, 768), device=DEVICE)
toy_biases = torch.tensor([[0.0, 1.0]] * K_FIXED, device=DEVICE)
toy_class_maps = torch.tensor([
    [LABEL_TO_ID[pair.split("|")[0]], LABEL_TO_ID[pair.split("|")[1]]]
    for pair in HARD_PAIRS
], dtype=torch.long, device=DEVICE)
toy_lookup = torch.full((len(LABELS), len(LABELS)), -1,
                        dtype=torch.long, device=DEVICE)
for toy_index, toy_pair in enumerate(HARD_PAIRS):
    toy_first, toy_second = [LABEL_TO_ID[x] for x in toy_pair.split("|")]
    toy_lookup[toy_first, toy_second] = toy_index
    toy_lookup[toy_second, toy_first] = toy_index
toy_heads = {
    "weights": toy_weights,
    "biases": toy_biases,
    "class_maps": toy_class_maps,
    "pair_lookup": toy_lookup,
}
toy_batch = {"input_ids": torch.zeros((1, 2), dtype=torch.long, device=DEVICE)}

toy_far_model = _ToyModel(
    toy_sequence,
    torch.tensor([[5.0, 0.0, -5.0, -5.0, -5.0]], device=DEVICE),
)
toy_a0_bypass = forward_system(toy_far_model, None, toy_batch, "A0_flat")
assert toy_a0_bypass["prediction"].item() == LABEL_TO_ID["Suicidal"]
toy_conditional_bypass = forward_system(
    toy_far_model, toy_heads, toy_batch, "A1_conditional"
)
assert not toy_conditional_bypass["active"].item()
toy_always_route = forward_system(
    toy_far_model, toy_heads, toy_batch, "A2_always_route"
)
assert toy_always_route["active"].item()
assert toy_always_route["prediction"].item() == LABEL_TO_ID["Depressed"]

toy_near_model = _ToyModel(
    toy_sequence,
    torch.tensor([[0.05, 0.0, -5.0, -5.0, -5.0]], device=DEVICE),
)
toy_conditional_route = forward_system(
    toy_near_model, toy_heads, toy_batch, "A1_conditional"
)
assert toy_conditional_route["active"].item()

toy_unmatched_model = _ToyModel(
    toy_sequence,
    torch.tensor([[-5.0, -5.0, 0.05, 0.0, -5.0]], device=DEVICE),
)
toy_unmatched_pair = forward_system(
    toy_unmatched_model, toy_heads, toy_batch, "A2_always_route"
)
assert not toy_unmatched_pair["active"].item()
assert toy_unmatched_pair["prediction"].item() == LABEL_TO_ID["Anxious"]
del toy_far_model, toy_near_model, toy_unmatched_model, toy_heads


print("PASS: frozen A0/A1/A2 forward implementation initialized")


PASS: frozen A0/A1/A2 forward implementation initialized


## Validation reproduction

All ten seeds reproduce Phase 10 A0/A1/A2 validation behavior before the notebook
is allowed to define the all-gates-passed state. This is the final implementation
check while test access remains false.


In [7]:
def strict_bool_array(series):
    normalized = series.astype(str).str.strip().str.lower()
    assert set(normalized.unique()).issubset({"true", "false"})
    return normalized.eq("true").to_numpy()


phase10_performance = pd.read_csv(PHASE10_ROOT / "phase10_control_seedwise.csv")
phase10_performance = phase10_performance[
    phase10_performance["control_id"].isin(SYSTEMS)
].rename(columns={"control_id": "system_id"})
assert len(phase10_performance) == 30


def reproduce_phase10_validation(model_seed):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRS[model_seed], local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIRS[model_seed], local_files_only=True, torch_dtype=torch.float32
    ).eval().to(DEVICE)
    heads = load_selected_heads(model_seed)
    collected = {system: [] for system in SYSTEMS}
    active = {"A1_conditional": [], "A2_always_route": []}
    pair_index = {"A1_conditional": [], "A2_always_route": []}
    top2_parts, margin_parts = [], []
    texts = validation["text"].astype(str).tolist()
    for start in range(0, len(texts), BATCH_SIZE):
        outputs = forward_all_systems(
            model, heads, gpu_batch(tokenizer, texts[start:start + BATCH_SIZE])
        )
        for system in SYSTEMS:
            collected[system].append(outputs[system]["prediction"].cpu())
        for system in ("A1_conditional", "A2_always_route"):
            active[system].append(outputs[system]["active"].cpu())
            pair_index[system].append(outputs[system]["pair_index"].cpu())
        top2_parts.append(outputs["A1_conditional"]["top2"].cpu())
        margin_parts.append(outputs["A1_conditional"]["margin"].cpu())
    inferred = {system: torch.cat(parts).numpy() for system, parts in collected.items()}
    inferred["top2"] = torch.cat(top2_parts).numpy()
    inferred["margin"] = torch.cat(margin_parts).numpy()
    max_margin_difference = 0.0
    max_metric_difference = 0.0
    for system in SYSTEMS:
        saved_path = PHASE10_ROOT / "controls" / system / f"seed_{model_seed}" / "validation_predictions.csv"
        saved = pd.read_csv(saved_path)
        assert np.array_equal(saved["original_row"].to_numpy(), validation["original_row"].to_numpy())
        assert np.array_equal(saved["final_id"].to_numpy(dtype=int), inferred[system])
        assert np.array_equal(saved["global_id"].to_numpy(dtype=int), inferred["A0_flat"])
        assert np.array_equal(saved["top1"].to_numpy(dtype=int), inferred["top2"][:, 0])
        assert np.array_equal(saved["top2"].to_numpy(dtype=int), inferred["top2"][:, 1])
        if system == "A0_flat":
            inferred_active = np.zeros(len(validation), dtype=bool)
            inferred_pairs = np.full(len(validation), "", dtype=object)
        else:
            inferred_active = torch.cat(active[system]).numpy().astype(bool)
            indices = torch.cat(pair_index[system]).numpy()
            inferred_pairs = np.array([
                HARD_PAIRS[int(value)] if is_active else ""
                for value, is_active in zip(indices, inferred_active)
            ], dtype=object)
        assert np.array_equal(strict_bool_array(saved["active"]), inferred_active)
        assert np.array_equal(saved["routed_pair"].fillna("").to_numpy(), inferred_pairs)
        difference = float(np.max(np.abs(saved["margin"].to_numpy(float) - inferred["margin"])))
        assert difference <= 1e-6
        max_margin_difference = max(max_margin_difference, difference)
        saved_metric = phase10_performance[
            (phase10_performance["model_seed"] == model_seed)
            & (phase10_performance["system_id"] == system)
        ].iloc[0]
        calculated = classification_metrics(y_validation, inferred[system], len(LABELS))
        for metric_name in ("accuracy", "macro_f1", "balanced_accuracy"):
            metric_difference = abs(
                float(calculated[metric_name]) - float(saved_metric[metric_name])
            )
            assert metric_difference <= 1e-12
            max_metric_difference = max(max_metric_difference, metric_difference)
        assert int(inferred_active.sum()) == int(saved_metric["routed_n"])
        assert abs(float(inferred_active.mean()) - float(saved_metric["routing_coverage"])) <= 1e-12
    del heads, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "model_seed": model_seed,
        "margin_max_abs_diff": max_margin_difference,
        "metric_max_abs_diff": max_metric_difference,
        "routing_count_exact": True,
        "routing_coverage_exact": True,
        "status": "PASS",
    }


reproduction_rows = []
for seed_index, model_seed in enumerate(ALL_SEEDS, start=1):
    print(f"validation reproduction [{seed_index}/10] seed {model_seed}")
    reproduction_rows.append(reproduce_phase10_validation(model_seed))
phase14_validation_reproduction_seedwise = pd.DataFrame(reproduction_rows)
assert len(phase14_validation_reproduction_seedwise) == 10
assert phase14_validation_reproduction_seedwise["status"].eq("PASS").all()
ALL_PREOPENING_GATES_PASSED = True
print("PASS: 10/10 Phase 10 validation reproductions")
print("Test data accessed: False")


validation reproduction [1/10] seed 13


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

validation reproduction [2/10] seed 42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

validation reproduction [9/10] seed 2026


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

validation reproduction [10/10] seed 3407


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PASS: 10/10 Phase 10 validation reproductions
Test data accessed: False


## Irreversible test-opening event

**Running the next cell opens the locked test.** It is permitted only after the
ten validation reproductions and every Phase 8–12 integrity gate pass. First this
cell writes and hashes the immutable internal specification and one-use opening
authorization. Only then is the test filename discovered. The opening record is
written immediately and blocks a second opening in the same session.


In [8]:
PREOPENING_SPEC_PATH = OUTPUT_DIR / "phase14_preopening_spec.json"
PREOPENING_SPEC_HASH_PATH = OUTPUT_DIR / "phase14_preopening_spec.sha256"
OPENING_AUTHORIZATION_PATH = OUTPUT_DIR / "phase14_test_opening_authorization.json"


def build_preopening_spec():
    return {
        "phase": 14,
        "freeze_event": "internal_preopening_protocol_freeze",
        "freeze_created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_seeds": ALL_SEEDS,
        "systems": SYSTEMS,
        "variants": VARIANTS,
        "labels": LABELS,
        "H": HARD_PAIRS,
        "K": K_FIXED,
        "conditional_tau": CONDITIONAL_TAU,
        "always_route_tau": ALWAYS_ROUTE_TAU,
        "maximum_length": MAX_LENGTH,
        "padding": "dynamic",
        "precision": "float32",
        "automatic_mixed_precision": False,
        "bootstrap_reps": BOOTSTRAP_REPS,
        "bootstrap_method": "class_stratified_paired_rows",
        "mcnemar_method": "exact_two_sided",
        "holm_family_sizes": EXPECTED_HOLM_FAMILY_SIZES,
        "primary_comparison": "A1_conditional_vs_A0_flat_original_macro_f1",
        "secondary_comparison": "A2_always_route_vs_A0_flat_original_macro_f1",
        "lexical_mask_version": "phase11-frozen-v1",
        "lexical_mask_spec_sha256": LEXICAL_MASK_SPEC_DIGEST,
        "validation_split": {
            "filename": VALIDATION_FILENAME,
            "sha256": VALIDATION_DIGEST,
            "row_count": len(validation),
            "required_columns": ["original_row", "text", "label"],
        },
        "test_identity_policy": {
            "filename": TEST_FILENAME,
            "required_columns": ["original_row", "text", "label"],
            "historical_test_hash_available": False,
            "historical_test_hash_limitation": (
                "The original dataset-lock manifest was unavailable; the observed "
                "test hash is recorded at the one-time opening before inference."
            ),
        },
        "upstream_artifacts": UPSTREAM_ARTIFACTS,
        "selected_heads": FROZEN_SELECTED_HEADS,
        "model_runs": FROZEN_MODEL_RUNS,
        "dataset_provenance": {
            "new_reaudit_performed": False,
            "treated_as_canonical_real_dataset": True,
        },
        "training_performed": False,
        "tuning_performed": False,
        "selection_performed": False,
        "test_outcomes_seen": False,
    }


assert ALL_PREOPENING_GATES_PASSED is True
assert phase14_validation_reproduction_seedwise["status"].eq("PASS").all()
FROZEN_PROTOCOL_PAYLOAD = build_preopening_spec()
write_json(PREOPENING_SPEC_PATH, FROZEN_PROTOCOL_PAYLOAD)
PREOPENING_SPEC_DIGEST = sha256_file(PREOPENING_SPEC_PATH)
PREOPENING_SPEC_HASH_PATH.write_text(PREOPENING_SPEC_DIGEST + "\n", encoding="utf-8")
OPENING_AUTHORIZATION = {
    "authorized": True,
    "test_open_count_allowed": 1,
    "test_outcomes_seen": False,
    "preopening_spec_sha256": PREOPENING_SPEC_DIGEST,
}
write_json(OPENING_AUTHORIZATION_PATH, OPENING_AUTHORIZATION)
OPENING_AUTHORIZATION_DIGEST = sha256_file(OPENING_AUTHORIZATION_PATH)
assert json.loads(PREOPENING_SPEC_PATH.read_text()) == FROZEN_PROTOCOL_PAYLOAD
assert PREOPENING_SPEC_HASH_PATH.read_text().strip() == PREOPENING_SPEC_DIGEST
assert json.loads(OPENING_AUTHORIZATION_PATH.read_text()) == OPENING_AUTHORIZATION
PREOPENING_FREEZE_COMPLETE = True
FROZEN_PROTOCOL_FINGERPRINT = PREOPENING_SPEC_DIGEST
print("PASS: immutable internal pre-opening freeze written and hashed")


def open_locked_test_once():
    assert ALL_PREOPENING_GATES_PASSED is True
    assert phase14_validation_reproduction_seedwise["status"].eq("PASS").all()
    assert PREOPENING_FREEZE_COMPLETE is True
    assert sha256_file(PREOPENING_SPEC_PATH) == PREOPENING_SPEC_DIGEST
    assert PREOPENING_SPEC_HASH_PATH.read_text().strip() == PREOPENING_SPEC_DIGEST
    assert OPENING_AUTHORIZATION["authorized"] is True
    assert OPENING_AUTHORIZATION["test_open_count_allowed"] == 1
    assert OPENING_AUTHORIZATION["test_outcomes_seen"] is False
    assert OPENING_AUTHORIZATION["preopening_spec_sha256"] == PREOPENING_SPEC_DIGEST
    if OPENING_RECORD_PATH.exists():
        raise RuntimeError("Locked test has already been opened in this session")
    test_policy = FROZEN_PROTOCOL_PAYLOAD["test_identity_policy"]
    assert test_policy["filename"] == TEST_FILENAME
    assert test_policy["historical_test_hash_available"] is False
    candidates = sorted(INPUT_ROOT.rglob(TEST_FILENAME), key=str)
    if not candidates:
        raise FileNotFoundError(TEST_FILENAME)
    candidate_digests = [sha256_file(path) for path in candidates]
    if len(set(candidate_digests)) != 1:
        raise ValueError(f"Conflicting duplicate inputs for {TEST_FILENAME}")
    test_path = candidates[0]
    observed_digest = candidate_digests[0]
    frame = pd.read_csv(test_path)
    assert len(frame) > 0
    required_columns = set(test_policy["required_columns"])
    assert {"original_row", "text", "label"}.issubset(required_columns)
    assert required_columns.issubset(frame.columns)
    assert frame["original_row"].is_unique
    assert set(frame["label"]) == set(LABELS)
    opened_at = datetime.now(timezone.utc).isoformat()
    record = {
        "phase": 14,
        "event": "one_time_locked_test_opening",
        "opened_at_utc": opened_at,
        "test_opened": True,
        "test_sha256": observed_digest,
        "observed_test_sha256": observed_digest,
        "test_rows": len(frame),
        "historical_test_hash_available": False,
        "preopening_spec_sha256": PREOPENING_SPEC_DIGEST,
        "authorization_sha256": OPENING_AUTHORIZATION_DIGEST,
        "protocol_fingerprint": FROZEN_PROTOCOL_FINGERPRINT,
    }
    write_json(OPENING_RECORD_PATH, record)
    record_input(
        test_path, "locked_test", access_stage="test_opening",
        known_sha256=observed_digest,
    )
    return test_path, frame, record


TEST_PATH, test_frame, TEST_OPENING_RECORD = open_locked_test_once()
y_test = test_frame["label"].map(LABEL_TO_ID).to_numpy(dtype=int)
assert len(y_test) == TEST_OPENING_RECORD["test_rows"]
print("LOCKED TEST OPENED ONCE:", TEST_OPENING_RECORD["opened_at_utc"])
print("Test SHA-256:", TEST_OPENING_RECORD["test_sha256"])
print("Test rows:", len(test_frame))


PASS: immutable internal pre-opening freeze written and hashed
LOCKED TEST OPENED ONCE: 2026-08-23T09:53:53.228573+00:00
Test SHA-256: e3a61197d1d4a57e61269be260570cc5aa5b504275f0370b1c379f0453517270
Test rows: 1491


## Frozen lexical conditions

Phase 11's byte-verified `phase11-frozen-v1` mask specification produces original,
direct-mask, and cumulative operational-mask text in memory. Only row IDs and mask
audit counts are retained; neither original nor masked text is saved.


In [9]:
assert MASK_SPEC["operational_is_cumulative"] is True
source_texts = test_frame["text"].astype(str).tolist()
variant_texts = {"original": list(source_texts)}
mask_audit_rows = []
for mode, variant in (("direct", "direct_mask"), ("operational", "operational_mask")):
    masked_texts = []
    for original_row, text in zip(test_frame["original_row"], source_texts):
        result = mask_text(text, MASK_SPEC, mode)
        masked_texts.append(result["text"])
        mask_audit_rows.append({
            "original_row": original_row,
            "variant": variant,
            "affected": result["affected"],
            "span_count": result["span_count"],
            "cue_ids": result["cue_ids"],
            "original_characters": result["original_characters"],
            "masked_characters": result["masked_characters"],
        })
    variant_texts[variant] = masked_texts
phase14_lexical_mask_audit = pd.DataFrame(mask_audit_rows)
assert set(variant_texts) == set(VARIANTS)
assert len(phase14_lexical_mask_audit) == 2 * len(test_frame)
assert not any(
    marker in column.lower()
    for column in phase14_lexical_mask_audit.columns
    for marker in ("source_text", "masked_text", "raw_text")
)
display(
    phase14_lexical_mask_audit.groupby("variant", as_index=False)
    .agg(rows=("original_row", "size"), affected=("affected", "sum"), spans=("span_count", "sum"))
)
print("PASS: three frozen text variants prepared without saved raw text")


,variant,rows,affected,spans
0,direct_mask,1491,422,859
1,operational_mask,1491,747,1914


PASS: three frozen text variants prepared without saved raw text


## Frozen test inference

One model seed is loaded at a time. Each text variant receives one global forward,
then the three frozen system outputs are derived from the shared representation.
Saved predictions contain identifiers and numeric evidence but no text.


In [10]:
prediction_rows = []
probability_columns = [f"global_p{index}" for index in range(len(LABELS))]
for seed_index, model_seed in enumerate(ALL_SEEDS, start=1):
    print(f"test inference [{seed_index}/10] seed {model_seed}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRS[model_seed], local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIRS[model_seed], local_files_only=True, torch_dtype=torch.float32
    ).eval().to(DEVICE)
    heads = load_selected_heads(model_seed)
    for variant in VARIANTS:
        texts = variant_texts[variant]
        for start in range(0, len(texts), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(texts))
            outputs = forward_all_systems(
                model, heads, gpu_batch(tokenizer, texts[start:stop])
            )
            probabilities = outputs["global_probabilities"].cpu().numpy()
            route_details = outputs["A1_conditional"]
            top2 = route_details["top2"].cpu().numpy()
            margin = route_details["margin"].cpu().numpy()
            global_prediction = outputs["A0_flat"]["prediction"].cpu().numpy()
            for local_index, frame_position in enumerate(range(start, stop)):
                base = {
                    "model_seed": model_seed,
                    "variant": variant,
                    "validation_position": frame_position,
                    "original_row": test_frame.iloc[frame_position]["original_row"],
                    "true_id": int(y_test[frame_position]),
                    "true_label": test_frame.iloc[frame_position]["label"],
                    "global_id": int(global_prediction[local_index]),
                    "top1": int(top2[local_index, 0]),
                    "top2": int(top2[local_index, 1]),
                    "margin": float(margin[local_index]),
                    "global_confidence": float(probabilities[local_index].max()),
                }
                for system_id in SYSTEMS:
                    details = outputs[system_id]
                    if system_id == "A0_flat":
                        is_active = False
                        routed_pair = ""
                    else:
                        is_active = bool(details["active"][local_index].item())
                        pair_value = int(details["pair_index"][local_index].item())
                        routed_pair = HARD_PAIRS[pair_value] if is_active else ""
                    row = {
                        **base,
                        "system_id": system_id,
                        "final_id": int(details["prediction"][local_index].item()),
                        "active": is_active,
                        "routed_pair": routed_pair,
                    }
                    for probability_index, column in enumerate(probability_columns):
                        row[column] = (
                            float(probabilities[local_index, probability_index])
                            if system_id == "A0_flat" else np.nan
                        )
                    prediction_rows.append(row)
    del heads, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  PASS: seed {model_seed} all variants")

phase14_test_predictions = pd.DataFrame(prediction_rows)
expected_prediction_rows = len(ALL_SEEDS) * len(VARIANTS) * len(SYSTEMS) * len(test_frame)
assert len(phase14_test_predictions) == expected_prediction_rows
assert not phase14_test_predictions.duplicated(
    ["model_seed", "variant", "system_id", "original_row"]
).any()
assert set(phase14_test_predictions["model_seed"]) == set(ALL_SEEDS)
assert set(phase14_test_predictions["system_id"]) == set(SYSTEMS)
assert set(phase14_test_predictions["variant"]) == set(VARIANTS)
assert not any("text" in column.lower() for column in phase14_test_predictions.columns)
print("PASS: frozen test inference complete; rows:", len(phase14_test_predictions))


test inference [1/10] seed 13


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: seed 13 all variants
test inference [2/10] seed 42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: seed 42 all variants
test inference [3/10] seed 77


...

  PASS: seed 1024 all variants
test inference [9/10] seed 2026


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: seed 2026 all variants
test inference [10/10] seed 3407


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: seed 3407 all variants
PASS: frozen test inference complete; rows: 134190


## Frozen metric helpers

These functions preserve Phase 11's BCR, calibration, global-confidence
risk-coverage, and AURC definitions. CTPR calibration is not reported because routed
predictions do not define replacement five-class probability distributions.


In [11]:
def compute_bcr(confusion):
    support = confusion.sum(axis=1)
    rows = []
    for first, second in itertools.combinations(range(len(LABELS)), 2):
        first_rate = confusion[first, second] / support[first] if support[first] else np.nan
        second_rate = confusion[second, first] / support[second] if support[second] else np.nan
        rows.append({
            "pair": f"{LABELS[first]}|{LABELS[second]}",
            "first_support": int(support[first]),
            "second_support": int(support[second]),
            "first_to_second": int(confusion[first, second]),
            "second_to_first": int(confusion[second, first]),
            "pair_errors": int(confusion[first, second] + confusion[second, first]),
            "bcr": float(0.5 * (first_rate + second_rate)),
        })
    return rows


def ece_table(probabilities, truth, bins=ECE_BINS):
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    assignments = np.minimum((confidence * bins).astype(int), bins - 1)
    rows, ece = [], 0.0
    for bin_id in range(bins):
        selected = assignments == bin_id
        count = int(selected.sum())
        if count:
            mean_confidence = float(confidence[selected].mean())
            accuracy = float((predicted[selected] == truth[selected]).mean())
            gap = abs(mean_confidence - accuracy)
            ece += count / len(truth) * gap
        else:
            mean_confidence = accuracy = gap = np.nan
        rows.append({
            "bin_id": bin_id, "lower": bin_id / bins,
            "upper": (bin_id + 1) / bins, "count": count,
            "mean_confidence": mean_confidence,
            "empirical_accuracy": accuracy, "absolute_gap": gap,
        })
    return float(ece), rows


def reliability_metrics(probabilities, truth):
    ece, bins = ece_table(probabilities, truth)
    one_hot = np.eye(len(LABELS))[truth]
    chosen = probabilities[np.arange(len(truth)), truth]
    return {
        "ece": ece,
        "nll": float(-np.mean(np.log(np.clip(chosen, NLL_EPSILON, 1.0)))),
        "multiclass_brier": float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))),
        "bins": bins,
    }


def risk_coverage_curve(truth, prediction, score, original_rows):
    order = np.lexsort((np.asarray(original_rows), -np.asarray(score)))
    errors = np.cumsum(np.asarray(truth)[order] != np.asarray(prediction)[order])
    total = len(order)
    return [{
        "retained_n": position,
        "coverage": position / total,
        "risk": float(errors[position - 1] / position),
        "threshold_score": float(np.asarray(score)[index]),
        "original_row": np.asarray(original_rows)[index].item()
        if hasattr(np.asarray(original_rows)[index], "item")
        else np.asarray(original_rows)[index],
    } for position, index in enumerate(order, start=1)]


print("PASS: frozen mechanism metric helpers initialized")


PASS: frozen mechanism metric helpers initialized


## Classification, routing, and mechanism results

Every seed-system-variant unit produces complete classification, routing,
correction/harm, class, pair, Suicidal false-negative, and BCR evidence.


In [12]:
metric_rows, per_class_rows, confusion_rows = [], [], []
routing_rows, correction_rows, correction_pair_rows, correction_class_rows = [], [], [], []
suicidal_rows, bcr_rows = [], []
grouped = phase14_test_predictions.groupby(["model_seed", "variant", "system_id"], sort=False)
for (model_seed, variant, system_id), group in grouped:
    group = group.sort_values("validation_position")
    truth = group["true_id"].to_numpy(int)
    prediction = group["final_id"].to_numpy(int)
    global_prediction = group["global_id"].to_numpy(int)
    active = group["active"].to_numpy(bool)
    metrics = classification_metrics(truth, prediction, len(LABELS))
    metric_rows.append({
        "model_seed": model_seed, "variant": variant, "system_id": system_id,
        "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
    })
    for class_id in range(len(LABELS)):
        per_class_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id,
            "class_id": class_id, "class_label": LABELS[class_id],
            "precision": metrics["precision"][class_id],
            "recall": metrics["recall"][class_id],
            "f1": metrics["f1"][class_id],
            "support": int(metrics["support"][class_id]),
        })
        for predicted_id in range(len(LABELS)):
            confusion_rows.append({
                "model_seed": model_seed, "variant": variant, "system_id": system_id,
                "true_id": class_id, "predicted_id": predicted_id,
                "count": int(metrics["confusion"][class_id, predicted_id]),
            })
    routing_rows.append({
        "model_seed": model_seed, "variant": variant, "system_id": system_id,
        "routed_n": int(active.sum()), "routing_coverage": float(active.mean()),
    })
    changes = correction_harm_counts(truth, global_prediction, prediction, active)
    correction_rows.append({
        "model_seed": model_seed, "variant": variant, "system_id": system_id, **changes
    })
    changed = active & (global_prediction != prediction)
    categories = np.full(len(group), "unchanged", dtype=object)
    categories[changed & (global_prediction != truth) & (prediction == truth)] = "correction"
    categories[changed & (global_prediction == truth) & (prediction != truth)] = "harm"
    categories[changed & (global_prediction != truth) & (prediction != truth)] = "neutral_change"
    detail = group[["routed_pair", "true_id"]].copy()
    detail["category"] = categories
    for pair_name, pair_group in detail[detail["routed_pair"] != ""].groupby("routed_pair"):
        counts = pair_group["category"].value_counts()
        correction_pair_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id,
            "routed_pair": pair_name, "routed_n": len(pair_group),
            "corrections": int(counts.get("correction", 0)),
            "harms": int(counts.get("harm", 0)),
            "neutral_changes": int(counts.get("neutral_change", 0)),
        })
    for class_id, class_group in detail.groupby("true_id"):
        counts = class_group["category"].value_counts()
        correction_class_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id,
            "true_id": int(class_id), "true_label": LABELS[int(class_id)],
            "corrections": int(counts.get("correction", 0)),
            "harms": int(counts.get("harm", 0)),
            "neutral_changes": int(counts.get("neutral_change", 0)),
        })
    suicidal = truth == LABEL_TO_ID["Suicidal"]
    false_negative = suicidal & (prediction != LABEL_TO_ID["Suicidal"])
    destinations = np.bincount(prediction[false_negative], minlength=len(LABELS))
    for destination_id in range(len(LABELS)):
        suicidal_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id,
            "destination_id": destination_id, "destination_label": LABELS[destination_id],
            "false_negative_count": int(destinations[destination_id]),
            "total_suicidal_false_negatives": int(false_negative.sum()),
        })
    for bcr in compute_bcr(metrics["confusion"]):
        bcr_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id, **bcr
        })

phase14_seedwise_metrics = pd.DataFrame(metric_rows)
phase14_per_class_metrics = pd.DataFrame(per_class_rows)
phase14_confusion_matrices = pd.DataFrame(confusion_rows)
phase14_routing_seedwise = pd.DataFrame(routing_rows)
phase14_correction_harm_seedwise = pd.DataFrame(correction_rows)
phase14_correction_harm_by_pair = pd.DataFrame(correction_pair_rows)
phase14_correction_harm_by_class = pd.DataFrame(correction_class_rows)
phase14_suicidal_false_negatives = pd.DataFrame(suicidal_rows)
phase14_bcr_seedwise = pd.DataFrame(bcr_rows)
assert len(phase14_seedwise_metrics) == 90
print("PASS: classification and mechanism tables complete")


PASS: classification and mechanism tables complete


## Reliability and risk-coverage

A0 alone receives ECE, NLL, and multiclass Brier measurements. All three systems use
frozen global maximum-softmax confidence for risk-coverage and AURC.


In [13]:
reliability_rows, reliability_bin_rows = [], []
risk_rows, aurc_rows = [], []
for (model_seed, variant), group in phase14_test_predictions[
    phase14_test_predictions["system_id"] == "A0_flat"
].groupby(["model_seed", "variant"], sort=False):
    group = group.sort_values("validation_position")
    probabilities = group[probability_columns].to_numpy(float)
    truth = group["true_id"].to_numpy(int)
    reliability = reliability_metrics(probabilities, truth)
    reliability_rows.append({
        "model_seed": model_seed, "variant": variant, "system_id": "A0_flat",
        "ece": reliability["ece"], "nll": reliability["nll"],
        "multiclass_brier": reliability["multiclass_brier"],
        "high_confidence_error_count": int(
            (
                (probabilities.max(axis=1) >= HIGH_CONFIDENCE_THRESHOLD)
                & (probabilities.argmax(axis=1) != truth)
            ).sum()
        ),
    })
    for bin_row in reliability["bins"]:
        reliability_bin_rows.append({
            "model_seed": model_seed, "variant": variant,
            "system_id": "A0_flat", **bin_row,
        })
for (model_seed, variant, system_id), group in grouped:
    group = group.sort_values("validation_position")
    curve = risk_coverage_curve(
        group["true_id"].to_numpy(int), group["final_id"].to_numpy(int),
        group["global_confidence"].to_numpy(float), group["original_row"].to_numpy(),
    )
    for row in curve:
        risk_rows.append({
            "model_seed": model_seed, "variant": variant, "system_id": system_id, **row
        })
    aurc_rows.append({
        "model_seed": model_seed, "variant": variant, "system_id": system_id,
        "aurc": float(np.mean([row["risk"] for row in curve])),
    })
phase14_reliability_seedwise = pd.DataFrame(reliability_rows)
phase14_reliability_bins = pd.DataFrame(reliability_bin_rows)
phase14_risk_coverage = pd.DataFrame(risk_rows)
phase14_aurc_seedwise = pd.DataFrame(aurc_rows)
assert len(phase14_reliability_seedwise) == 30
assert len(phase14_aurc_seedwise) == 90
print("PASS: reliability and risk-coverage tables complete")


PASS: reliability and risk-coverage tables complete


## Paired statistical inference

Macro-F1 changes use 5,000 class-stratified paired resamples. Exact two-sided
McNemar p-values are corrected within the three frozen Holm families of sizes 10,
10, and 40. Across-seed effects remain descriptive.


In [14]:
bootstrap_rows, mcnemar_rows = [], []
for variant_index, variant in enumerate(VARIANTS):
    for seed_index, model_seed in enumerate(ALL_SEEDS):
        selected = phase14_test_predictions[
            (phase14_test_predictions["variant"] == variant)
            & (phase14_test_predictions["model_seed"] == model_seed)
        ]
        flat = selected[selected["system_id"] == "A0_flat"].sort_values("validation_position")
        truth = flat["true_id"].to_numpy(int)
        flat_prediction = flat["final_id"].to_numpy(int)
        for system_index, system_id in enumerate(("A1_conditional", "A2_always_route"), start=1):
            system = selected[selected["system_id"] == system_id].sort_values("validation_position")
            prediction = system["final_id"].to_numpy(int)
            if variant == "original" and system_id == "A1_conditional":
                family_id = "primary_original_a1"
            elif variant == "original" and system_id == "A2_always_route":
                family_id = "secondary_original_a2"
            else:
                family_id = "robustness_masked"
            inference_seed = BOOTSTRAP_SEED + variant_index * 1000 + seed_index * 10 + system_index
            bootstrap = paired_macro_f1_bootstrap(
                truth, prediction, flat_prediction, class_count=len(LABELS),
                reps=BOOTSTRAP_REPS, seed=inference_seed,
            )
            bootstrap_rows.append({
                "model_seed": model_seed, "variant": variant,
                "system_id": system_id, "reference_system": "A0_flat",
                "family_id": family_id, **bootstrap,
            })
            mcnemar = exact_mcnemar(prediction == truth, flat_prediction == truth)
            mcnemar_rows.append({
                "model_seed": model_seed, "variant": variant,
                "system_id": system_id, "reference_system": "A0_flat",
                "family_id": family_id, **mcnemar,
            })
phase14_paired_bootstrap = pd.DataFrame(bootstrap_rows)
phase14_mcnemar_holm = pd.DataFrame(mcnemar_rows)
for family_id, expected_size in EXPECTED_HOLM_FAMILY_SIZES.items():
    selected = phase14_mcnemar_holm["family_id"] == family_id
    assert int(selected.sum()) == expected_size
    phase14_mcnemar_holm.loc[selected, "holm_adjusted_p"] = holm_adjust(
        phase14_mcnemar_holm.loc[selected, "p_value"].to_numpy(float)
    )
assert len(phase14_paired_bootstrap) == 60
assert phase14_paired_bootstrap["replicates"].eq(BOOTSTRAP_REPS).all()
assert phase14_mcnemar_holm["holm_adjusted_p"].notna().all()
print("PASS: paired bootstrap and frozen Holm families complete")


PASS: paired bootstrap and frozen Holm families complete


## Equal-seed summaries and claim guardrails

Summary statistics give each fine-tuning seed equal weight. Guardrails retain null
and negative results and prohibit directional improvement language when the paired
confidence interval includes zero.


In [15]:
summary_rows = []
for (variant, system_id), group in phase14_seedwise_metrics.groupby(["variant", "system_id"], sort=False):
    for metric_name in ("accuracy", "macro_f1", "balanced_accuracy"):
        summary_rows.append({
            "variant": variant, "system_id": system_id, "metric": metric_name,
            **equal_seed_descriptives(group[metric_name].to_numpy(float)),
        })
for (variant, system_id), group in phase14_paired_bootstrap.groupby(
    ["variant", "system_id"], sort=False
):
    summary_rows.append({
        "variant": variant,
        "system_id": system_id,
        "metric": "macro_f1_delta_vs_flat",
        **equal_seed_descriptives(group["observed_delta"].to_numpy(float)),
    })
phase14_equal_seed_summary = pd.DataFrame(summary_rows)

original = phase14_seedwise_metrics[phase14_seedwise_metrics["variant"] == "original"]
lexical_rows = []
for row in phase14_seedwise_metrics.itertuples(index=False):
    baseline = original[
        (original["model_seed"] == row.model_seed)
        & (original["system_id"] == row.system_id)
    ].iloc[0]
    lexical_rows.append({
        "model_seed": row.model_seed, "variant": row.variant, "system_id": row.system_id,
        "accuracy": row.accuracy, "macro_f1": row.macro_f1,
        "balanced_accuracy": row.balanced_accuracy,
        "delta_accuracy_vs_original": row.accuracy - baseline.accuracy,
        "delta_macro_f1_vs_original": row.macro_f1 - baseline.macro_f1,
        "delta_balanced_accuracy_vs_original": row.balanced_accuracy - baseline.balanced_accuracy,
    })
phase14_lexical_robustness_seedwise = pd.DataFrame(lexical_rows)
lexical_summary_rows = []
for (variant, system_id), group in phase14_lexical_robustness_seedwise.groupby(["variant", "system_id"], sort=False):
    for metric_name in ("delta_accuracy_vs_original", "delta_macro_f1_vs_original", "delta_balanced_accuracy_vs_original"):
        lexical_summary_rows.append({
            "variant": variant, "system_id": system_id, "metric": metric_name,
            **equal_seed_descriptives(group[metric_name].to_numpy(float)),
        })
phase14_lexical_robustness_summary = pd.DataFrame(lexical_summary_rows)
guardrail_rows = []
for row in phase14_paired_bootstrap.itertuples(index=False):
    if row.ci_low > 0:
        interval_direction = "positive"
    elif row.ci_high < 0:
        interval_direction = "negative"
    else:
        interval_direction = "includes_zero"
    adjusted_p = phase14_mcnemar_holm[
        (phase14_mcnemar_holm["model_seed"] == row.model_seed)
        & (phase14_mcnemar_holm["variant"] == row.variant)
        & (phase14_mcnemar_holm["system_id"] == row.system_id)
    ]["holm_adjusted_p"].iloc[0]
    guardrail_rows.append({
        "model_seed": row.model_seed, "variant": row.variant,
        "system_id": row.system_id, "macro_f1_delta": row.observed_delta,
        "ci_low": row.ci_low, "ci_high": row.ci_high,
        "interval_direction": interval_direction,
        "holm_adjusted_mcnemar_p": adjusted_p,
        "directional_improvement_claim_supported": interval_direction == "positive",
    })
phase14_claim_guardrails = pd.DataFrame(guardrail_rows)
display(phase14_equal_seed_summary.round(5))
print("PASS: equal-seed summaries and claim guardrails complete")


,variant,system_id,metric,count,mean,sd,median,q1,q3,iqr,minimum,maximum,positive_fraction
0,original,A0_flat,accuracy,10,0.84675,0.00454,0.84641,0.84390,0.84993,0.00604,0.84038,0.85312,1.0
1,original,A0_flat,macro_f1,10,0.84394,0.00467,0.84473,0.83935,0.84790,0.00855,0.83739,0.84973,1.0
2,original,A0_flat,balanced_accuracy,10,0.84647,0.00492,0.84640,0.84291,0.85006,0.00715,0.83944,0.85470,1.0
3,original,A1_conditional,accuracy,10,0.84735,0.00510,0.84809,0.84272,0.85111,0.00838,0.83970,0.85446,1.0
4,original,A1_conditional,macro_f1,10,0.84461,0.00517,0.84572,0.83952,0.84919,0.00966,0.83737,0.85123,1.0
5,original,A1_conditional,balanced_accuracy,10,0.84715,0.00514,0.84756,0.84339,0.85079,0.00741,0.83832,0.85555,1.0
6,original,A2_always_route,accuracy,10,0.84735,0.00472,0.84909,0.84373,0.85044,0.00671,0.83970,0.85379,1.0
7,original,A2_always_route,macro_f1,10,0.84482,0.00478,0.84641,0.84111,0.84865,0.00754,0.83744,0.85092,1.0
8,original,A2_always_route,balanced_accuracy,10,0.84768,0.00439,0.84786,0.84391,0.85102,0.00711,0.84135,0.85443,1.0
9,direct_mask,A0_flat,accuracy,10,0.81066,0.00688,0.81221,0.80785,0.81573,0.00788,0.79678,0.81891,1.0


PASS: equal-seed summaries and claim guardrails complete


## Reviewer-facing figures

Figures show original Macro-F1, seedwise A1 changes, routing, correction/harm,
lexical stress effects, Suicidal false negatives, and paired confidence intervals.
Every figure is saved as PNG and SVG.


In [16]:
COLORS = {"A0_flat": "#4C78A8", "A1_conditional": "#59A14F", "A2_always_route": "#E15759"}
LABEL_NAMES = {"A0_flat": "Flat", "A1_conditional": "Conditional", "A2_always_route": "Always-route"}

def save_figure(figure, stem):
    figure.tight_layout()
    for extension in ("png", "svg"):
        figure.savefig(FIGURE_DIR / f"{stem}.{extension}", dpi=220, bbox_inches="tight")
    plt.close(figure)

original_summary = phase14_equal_seed_summary[
    (phase14_equal_seed_summary["variant"] == "original")
    & (phase14_equal_seed_summary["metric"] == "macro_f1")
].set_index("system_id").loc[SYSTEMS]
figure, axis = plt.subplots(figsize=(7.2, 4.6))
axis.bar(range(3), original_summary["mean"], yerr=original_summary["sd"],
         color=[COLORS[value] for value in SYSTEMS], capsize=4)
axis.set_xticks(range(3), [LABEL_NAMES[value] for value in SYSTEMS])
axis.set_ylabel("Locked-test Macro-F1")
axis.grid(axis="y", alpha=0.25)
save_figure(figure, "phase14_original_macro_f1")

primary = phase14_paired_bootstrap[
    (phase14_paired_bootstrap["variant"] == "original")
    & (phase14_paired_bootstrap["system_id"] == "A1_conditional")
].sort_values("model_seed")
figure, axis = plt.subplots(figsize=(8.0, 4.6))
axis.axhline(0, color="black", linewidth=1)
axis.errorbar(primary["model_seed"].astype(str), primary["observed_delta"],
              yerr=[primary["observed_delta"] - primary["ci_low"], primary["ci_high"] - primary["observed_delta"]],
              fmt="o", color=COLORS["A1_conditional"], capsize=3)
axis.set_ylabel("A1 − A0 Macro-F1")
axis.set_xlabel("Model seed")
save_figure(figure, "phase14_a1_seedwise_delta")

for stem, frame, value, ylabel in (
    ("phase14_routing_coverage", phase14_routing_seedwise, "routing_coverage", "Routing coverage"),
    ("phase14_suicidal_false_negatives", phase14_suicidal_false_negatives.groupby(["model_seed", "variant", "system_id"], as_index=False)["total_suicidal_false_negatives"].first(), "total_suicidal_false_negatives", "Suicidal false negatives"),
):
    selected = frame[frame["variant"] == "original"].groupby("system_id")[value].mean().reindex(SYSTEMS)
    figure, axis = plt.subplots(figsize=(7.2, 4.6))
    axis.bar(range(3), selected, color=[COLORS[item] for item in SYSTEMS])
    axis.set_xticks(range(3), [LABEL_NAMES[item] for item in SYSTEMS])
    axis.set_ylabel(ylabel)
    axis.grid(axis="y", alpha=0.25)
    save_figure(figure, stem)

changes = phase14_correction_harm_seedwise[
    (phase14_correction_harm_seedwise["variant"] == "original")
    & (phase14_correction_harm_seedwise["system_id"] != "A0_flat")
].groupby("system_id")[["corrections", "harms"]].mean().reindex(SYSTEMS[1:])
figure, axis = plt.subplots(figsize=(7.2, 4.6))
x = np.arange(2)
axis.bar(x - 0.18, changes["corrections"], 0.36, label="Corrections", color="#59A14F")
axis.bar(x + 0.18, changes["harms"], 0.36, label="Harms", color="#E15759")
axis.set_xticks(x, [LABEL_NAMES[item] for item in SYSTEMS[1:]])
axis.legend()
save_figure(figure, "phase14_corrections_harms")

masked = phase14_lexical_robustness_summary[
    (phase14_lexical_robustness_summary["metric"] == "delta_macro_f1_vs_original")
    & (phase14_lexical_robustness_summary["variant"] != "original")
]
figure, axis = plt.subplots(figsize=(8.0, 4.8))
for system_id in SYSTEMS:
    values = masked[masked["system_id"] == system_id].set_index("variant").reindex(VARIANTS[1:])
    axis.plot(VARIANTS[1:], values["mean"], marker="o", label=LABEL_NAMES[system_id], color=COLORS[system_id])
axis.axhline(0, color="black", linewidth=1)
axis.set_ylabel("Macro-F1 change vs original")
axis.legend()
save_figure(figure, "phase14_lexical_macro_f1_change")

figure, axis = plt.subplots(figsize=(8.0, 4.8))
for system_id in SYSTEMS[1:]:
    values = phase14_paired_bootstrap[
        (phase14_paired_bootstrap["variant"] == "original")
        & (phase14_paired_bootstrap["system_id"] == system_id)
    ]
    axis.scatter(values["observed_delta"], values["ci_high"] - values["ci_low"],
                 label=LABEL_NAMES[system_id], color=COLORS[system_id])
axis.axvline(0, color="black", linewidth=1)
axis.set_xlabel("Observed Macro-F1 difference vs Flat")
axis.set_ylabel("95% interval width")
axis.legend()
save_figure(figure, "phase14_paired_effect_intervals")
print("PASS: reviewer-facing figures saved")


PASS: reviewer-facing figures saved


## Export and completion

The final cell audits structure and raw-text exclusion, writes all evidence, creates
an exact SHA-256 manifest, rebuilds the deterministic archive twice, and prints the
completion banner only after every one-time test condition passes.


In [17]:
phase14_structure_audit = pd.DataFrame([
    {"check": "preopening_gates", "observed": int(ALL_PREOPENING_GATES_PASSED), "expected": 1, "passed": ALL_PREOPENING_GATES_PASSED},
    {"check": "preopening_freeze", "observed": int(PREOPENING_FREEZE_COMPLETE), "expected": 1, "passed": PREOPENING_FREEZE_COMPLETE},
    {"check": "preopening_spec_hash", "observed": sha256_file(PREOPENING_SPEC_PATH), "expected": PREOPENING_SPEC_DIGEST, "passed": sha256_file(PREOPENING_SPEC_PATH) == PREOPENING_SPEC_DIGEST},
    {"check": "validation_reproductions", "observed": int(phase14_validation_reproduction_seedwise["status"].eq("PASS").sum()), "expected": 10, "passed": phase14_validation_reproduction_seedwise["status"].eq("PASS").all()},
    {"check": "selected_heads", "observed": verified_selected_heads, "expected": 30, "passed": verified_selected_heads == 30},
    {"check": "prediction_rows", "observed": len(phase14_test_predictions), "expected": expected_prediction_rows, "passed": len(phase14_test_predictions) == expected_prediction_rows},
    {"check": "bootstrap_rows", "observed": len(phase14_paired_bootstrap), "expected": 60, "passed": len(phase14_paired_bootstrap) == 60},
    {"check": "holm_rows", "observed": len(phase14_mcnemar_holm), "expected": 60, "passed": len(phase14_mcnemar_holm) == 60},
    {"check": "test_opened_once", "observed": int(TEST_OPENING_RECORD["test_opened"]), "expected": 1, "passed": TEST_OPENING_RECORD["test_opened"] is True},
])
assert phase14_structure_audit["passed"].all()
forbidden_columns = {"text", "raw_text", "source_text", "masked_text"}
assert forbidden_columns.isdisjoint({column.lower() for column in phase14_test_predictions.columns})
assert forbidden_columns.isdisjoint({column.lower() for column in phase14_lexical_mask_audit.columns})

phase14_input_inventory = pd.DataFrame(ACCESSED_INPUTS)
frames = {
    "phase14_input_inventory.csv": phase14_input_inventory,
    "phase14_validation_reproduction_seedwise.csv": phase14_validation_reproduction_seedwise,
    "phase14_test_predictions_no_raw_text.csv": phase14_test_predictions,
    "phase14_seedwise_metrics.csv": phase14_seedwise_metrics,
    "phase14_equal_seed_summary.csv": phase14_equal_seed_summary,
    "phase14_per_class_metrics.csv": phase14_per_class_metrics,
    "phase14_confusion_matrices.csv": phase14_confusion_matrices,
    "phase14_routing_seedwise.csv": phase14_routing_seedwise,
    "phase14_correction_harm_seedwise.csv": phase14_correction_harm_seedwise,
    "phase14_correction_harm_by_pair.csv": phase14_correction_harm_by_pair,
    "phase14_correction_harm_by_class.csv": phase14_correction_harm_by_class,
    "phase14_suicidal_false_negatives.csv": phase14_suicidal_false_negatives,
    "phase14_bcr_seedwise.csv": phase14_bcr_seedwise,
    "phase14_reliability_seedwise.csv": phase14_reliability_seedwise,
    "phase14_reliability_bins.csv": phase14_reliability_bins,
    "phase14_risk_coverage.csv": phase14_risk_coverage,
    "phase14_aurc_seedwise.csv": phase14_aurc_seedwise,
    "phase14_lexical_mask_audit_no_raw_text.csv": phase14_lexical_mask_audit,
    "phase14_lexical_robustness_seedwise.csv": phase14_lexical_robustness_seedwise,
    "phase14_lexical_robustness_summary.csv": phase14_lexical_robustness_summary,
    "phase14_paired_bootstrap.csv": phase14_paired_bootstrap,
    "phase14_mcnemar_holm.csv": phase14_mcnemar_holm,
    "phase14_claim_guardrails.csv": phase14_claim_guardrails,
    "phase14_structure_audit.csv": phase14_structure_audit,
}

# Defence in depth: column-name checks above prevent text-bearing schemas, while
# this value-level audit detects an exact raw-post value in any non-categorical
# exported string cell. Only hashes are compared; raw posts are never written.
raw_text_hashes = {
    hashlib.sha256(value.encode("utf-8")).hexdigest()
    for value in source_texts if value
}
categorical_markers = (
    "label", "class", "system", "variant", "pair", "status", "comparison",
    "scope", "metric", "check", "direction", "verdict", "control",
    "family", "cue_ids",
)
raw_text_value_overlap = []
for filename, frame in frames.items():
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        if any(marker in column.lower() for marker in categorical_markers):
            continue
        for value in frame[column].dropna().astype(str):
            digest = hashlib.sha256(value.encode("utf-8")).hexdigest()
            if digest in raw_text_hashes:
                raw_text_value_overlap.append((filename, column))
assert not raw_text_value_overlap, raw_text_value_overlap
privacy_row = pd.DataFrame([{
    "check": "raw_text_value_overlap",
    "observed": len(raw_text_value_overlap),
    "expected": 0,
    "passed": not raw_text_value_overlap,
}])
phase14_structure_audit = pd.concat(
    [phase14_structure_audit, privacy_row], ignore_index=True
)
frames["phase14_structure_audit.csv"] = phase14_structure_audit
for filename, frame in frames.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, float_format="%.17g")

device_properties = torch.cuda.get_device_properties(0)
environment = {
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "cuda_runtime": torch.version.cuda, "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_memory_bytes": int(device_properties.total_memory),
    "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
    "precision": "float32", "automatic_mixed_precision": False,
}
write_json(OUTPUT_DIR / "phase14_environment.json", environment)
phase14_protocol = {
    **FROZEN_PROTOCOL_PAYLOAD,
    "protocol_fingerprint": FROZEN_PROTOCOL_FINGERPRINT,
    "test_sha256": TEST_OPENING_RECORD["test_sha256"],
    "test_rows": TEST_OPENING_RECORD["test_rows"],
    "test_opened_at_utc": TEST_OPENING_RECORD["opened_at_utc"],
    "training_performed": False, "tuning_performed": False,
    "selection_performed": False, "test_accessed": True,
    "dataset_provenance": FROZEN_PROTOCOL_PAYLOAD["dataset_provenance"],
    "historical_test_hash_available": False,
    "preopening_spec_sha256": PREOPENING_SPEC_DIGEST,
    "opening_authorization_sha256": OPENING_AUTHORIZATION_DIGEST,
}
write_json(OUTPUT_DIR / "phase14_protocol.json", phase14_protocol)
protocol_digest = sha256_file(OUTPUT_DIR / "phase14_protocol.json")
(OUTPUT_DIR / "phase14_protocol.sha256").write_text(protocol_digest + "\n", encoding="utf-8")
execution_summary = {
    "phase": 14, "status": "PASS", "seed_count": 10,
    "system_count": 3, "variant_count": 3,
    "test_rows": len(test_frame), "prediction_rows": len(phase14_test_predictions),
    "validation_reproduction_pass_count": 10,
    "bootstrap_rows": len(phase14_paired_bootstrap),
    "bootstrap_reps_per_comparison": BOOTSTRAP_REPS,
    "holm_family_sizes": EXPECTED_HOLM_FAMILY_SIZES,
    "test_accessed": True, "test_open_count": 1,
    "preopening_freeze_complete": PREOPENING_FREEZE_COMPLETE,
    "historical_test_hash_available": False,
}
write_json(OUTPUT_DIR / "phase14_execution_summary.json", execution_summary)
report_lines = [
    "# Phase 14 Final Locked-Test Report", "", "Status: PASS", "",
    f"Test opened at UTC: {TEST_OPENING_RECORD['opened_at_utc']}",
    f"Test SHA-256: {TEST_OPENING_RECORD['test_sha256']}",
    f"Seeds: {ALL_SEEDS}", f"Systems: {SYSTEMS}", f"Variants: {VARIANTS}",
    "", "## Equal-seed results", "", "```csv",
    phase14_equal_seed_summary.to_csv(index=False).strip(), "```", "",
    "## Claim guardrails", "", "```csv",
    phase14_claim_guardrails.to_csv(index=False).strip(), "```", "",
    "All negative and null outcomes are retained.",
    "Lexical masks are stress tests, not natural-distribution estimates.",
    "Seeds quantify fine-tuning variability and are not independent datasets.",
    "Dataset labels are operational labels, not clinical diagnoses.",
    "Results do not establish clinical safety or deployment readiness.",
    "The supplied dataset was treated as canonical without a new re-audit.",
    "The original dataset-lock manifest was unavailable; the observed test hash was recorded at opening before inference.",
]
(OUTPUT_DIR / "phase14_report.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

manifest_rows = []
for path in sorted(item for item in OUTPUT_DIR.rglob("*") if item.is_file()):
    manifest_rows.append({
        "path": path.relative_to(OUTPUT_DIR).as_posix(),
        "sha256": sha256_file(path), "bytes": path.stat().st_size,
    })
pd.DataFrame(manifest_rows).to_csv(OUTPUT_DIR / "sha256_manifest.csv", index=False)
verify_manifest(OUTPUT_DIR)
result_zip = WORKING_ROOT / "MentalDistress_Phase14_Final_Locked_Test_Results.zip"
check_zip = WORKING_ROOT / "phase14_determinism_check.zip"
zip_digest = deterministic_zip(OUTPUT_DIR, result_zip, OUTPUT_DIR.name)
assert zip_digest == deterministic_zip(OUTPUT_DIR, check_zip, OUTPUT_DIR.name)
check_zip.unlink()

print("=" * 76)
print("PHASE 14 COMPLETE")
print("Systems:", SYSTEMS)
print("Frozen H/K/tau:", HARD_PAIRS, K_FIXED, CONDITIONAL_TAU)
print("Always-route tau:", ALWAYS_ROUTE_TAU)
print("Seeds: 10 / 10")
print("Test SHA-256:", TEST_OPENING_RECORD["test_sha256"])
print("Test opened at:", TEST_OPENING_RECORD["opened_at_utc"])
print("Test rows:", len(test_frame))
print("Bootstrap replicates per comparison:", BOOTSTRAP_REPS)
print("Holm family sizes:", EXPECTED_HOLM_FAMILY_SIZES)
print("Pre-opening specification SHA-256:", PREOPENING_SPEC_DIGEST)
print("Historical test hash available: False")
print("Results ZIP:", result_zip)
print("ZIP SHA-256:", zip_digest)
print("=" * 76)


PHASE 14 COMPLETE
Systems: ['A0_flat', 'A1_conditional', 'A2_always_route']
Frozen H/K/tau: ['Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others'] 3 0.1
Always-route tau: 1.0
Seeds: 10 / 10
Test SHA-256: e3a61197d1d4a57e61269be260570cc5aa5b504275f0370b1c379f0453517270
Test opened at: 2026-08-23T09:53:53.228573+00:00
Test rows: 1491
Bootstrap replicates per comparison: 5000
Holm family sizes: {'primary_original_a1': 10, 'secondary_original_a2': 10, 'robustness_masked': 40}
Pre-opening specification SHA-256: b0908463ee806e11e4fd0086ec2e3b4de02e89f9023a2b03e309aa3eabc98885
Historical test hash available: False
Results ZIP: /kaggle/working/MentalDistress_Phase14_Final_Locked_Test_Results.zip
ZIP SHA-256: 4f83c651891650f2f8f9d54dda9088f0e0caf19ef4fb73de91ed1e9893e32e8b
